# Что на самом деле лежит в наших датасетах

Блокнот показывает **авторские аннотации** — то, что разметили создатели
датасетов, без наших производных меток. Плюс примеры кадров, размер и оценку
качества. В конце отдельной таблицей: что было у авторов и во что мы это
превратили.

Нужен, потому что производные метки успели запутать: мы дважды меняли правила,
и стало неочевидно, откуда что берётся.

## Важное различие

| | что это |
|---|---|
| **авторская аннотация** | то, что размечено создателями датасета; факт |
| **наша метка** | результат наших правил из `data/sources.yaml`; интерпретация |

Второе выводится из первого и уже один раз оказалось неверным. Смотреть надо
на первое.

## 1. Окружение, бакет, ClearML

In [ ]:
%pip install -q boto3 pyarrow pandas matplotlib pillow

import io, os, socket, tarfile, zipfile, datetime
import boto3, numpy as np, pandas as pd, pyarrow.parquet as pq

BUCKET   = os.environ.get("DATASETS_BUCKET", "occlusionnet-clearml-b052c3-datasets")
ENDPOINT = "https://storage.yandexcloud.net"
REGION   = "ru-central1"

missing = [k for k in ("S3_KEY", "S3_SECRET") if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"нет ключей {missing}: секреты проекта DataSphere и "
                       "перезапуск ядра, либо .env для локального ядра")

s3 = boto3.client("s3", endpoint_url=ENDPOINT, region_name=REGION,
                  aws_access_key_id=os.environ["S3_KEY"],
                  aws_secret_access_key=os.environ["S3_SECRET"])

objs = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET):
    objs.extend(page.get("Contents", []))

print("хост:", socket.gethostname())
for o in sorted(objs, key=lambda x: x["Key"]):
    print(f"  {o['Key']:<52} {o['Size']/1024**3:7.2f} ГБ")

## 2. Манифесты

Отсюда берутся размеры и метрики качества — они посчитаны один раз при заезде.
Наши метки в этой таблице тоже есть, но до раздела 8 мы на них не смотрим.

In [ ]:
frames = []
for o in objs:
    k = o["Key"]
    if k.startswith("manifest/") and k.endswith(".parquet") and "archive/" not in k:
        body = s3.get_object(Bucket=BUCKET, Key=k)["Body"].read()
        frames.append(pq.read_table(io.BytesIO(body)).to_pandas())

df = pd.concat(frames, ignore_index=True)
df = df[df.error.isna()].copy()
print(f"кадров: {len(df):,}   источников: {df.source.nunique()}")

## 3. Чтение кадров из архивов

Сырьё лежит архивами на 9 ГБ. Достаём отдельные файлы range-запросами: у zip
есть оглавление, у tar его нет, поэтому для tar один раз строим индекс.

In [ ]:
class S3File(io.RawIOBase):
    def __init__(self, client, bucket, key):
        self.c, self.b, self.k = client, bucket, key
        self.size = client.head_object(Bucket=bucket, Key=key)["ContentLength"]
        self.pos = 0
    def readable(self): return True
    def seekable(self): return True
    def tell(self):     return self.pos
    def seek(self, offset, whence=io.SEEK_SET):
        if   whence == io.SEEK_SET: self.pos = offset
        elif whence == io.SEEK_CUR: self.pos += offset
        else:                       self.pos = self.size + offset
        self.pos = max(0, min(self.pos, self.size)); return self.pos
    def read(self, n=-1):
        if n is None or n < 0: n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0: return b""
        r = self.c.get_object(Bucket=self.b, Key=self.k,
                              Range=f"bytes={self.pos}-{self.pos + n - 1}")
        d = r["Body"].read(); self.pos += len(d); return d
    def readinto(self, buf):
        d = self.read(len(buf)); buf[:len(d)] = d; return len(d)

_TAR = {}
def tar_index(key):
    if key in _TAR: return _TAR[key]
    raw = S3File(s3, BUCKET, key)              # без буферизации: seek не читает
    idx = {}
    with tarfile.open(fileobj=raw, mode="r:") as t:
        for m in t:
            if m.isfile(): idx[m.name] = (m.offset_data, m.size)
    _TAR[key] = idx
    print(f"индекс {key}: {len(idx):,} членов")
    return idx

def archive_for(row):
    if row.source == "evocargo_raindrops":
        return "raw/evocargo_raindrops/RaindropsOnWindshield.zip", row.raw_path
    if row.source == "cadc":
        return f"raw/cadc/{row.raw_path.split('/')[0]}.tar", row.raw_path
    raise KeyError(row.source)

def read_member(key, member):
    if key.endswith(".zip"):
        fh = io.BufferedReader(S3File(s3, BUCKET, key), buffer_size=1 << 20)
        try:
            with zipfile.ZipFile(fh) as z:
                try:                return z.read(member)
                except KeyError:
                    hit = next((n for n in z.namelist() if n.endswith(member)), None)
                    if hit is None: raise
                    return z.read(hit)
        finally:
            fh.close()
    idx = tar_index(key)
    if member not in idx:
        hit = next((n for n in idx if n.endswith(member)), None)
        if hit is None: raise KeyError(member)
        member = hit
    off, size = idx[member]
    raw = S3File(s3, BUCKET, key); raw.seek(off)
    return raw.read(size)

## 4. Как показываем кадр

Под каждой картинкой — размер, вес файла и две метрики качества из манифеста.

**Резкость** — дисперсия лапласиана, посчитанная на копии, приведённой к высоте
256 пикселей. Больше значит резче; смазанные и расфокусированные кадры дают
низкие значения. Нормировка по высоте обязательна, иначе кадры разного
разрешения несравнимы.

**Яркость** — средняя светлота, 0 чёрный, 255 белый. Рядом `p99`: если он
упёрся в 255, света выбиты.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

INK, MUTED, BLUE, ORANGE = "#0b0b0b", "#898781", "#2a78d6", "#eb6834"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
                     "text.color": INK, "figure.dpi": 110,
                     "axes.edgecolor": "#e1e0d9",
                     "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False})

def caption(r):
    return (f"{r.sequence_id}\n"
            f"{r.width}x{r.height} · {r.bytes/1024**2:.1f} МБ\n"
            f"резкость {r.sharpness:.0f} · яркость {r.luma_mean:.0f} (p99 {r.luma_p99:.0f})")

def grid(rows, title, max_side=460, overlay_mask=False):
    """Ряд кадров с подписями. overlay_mask — наложить авторскую маску Evocargo."""
    rows = rows.reset_index(drop=True)
    if rows.empty:
        print(title, "— нет кадров"); return None
    fig, axes = plt.subplots(1, len(rows), figsize=(3.6 * len(rows), 4.2))
    axes = [axes] if len(rows) == 1 else list(axes)
    for ax, (_, r) in zip(axes, rows.iterrows()):
        key, member = archive_for(r)
        img = Image.open(io.BytesIO(read_member(key, member))).convert("RGB")
        img.thumbnail((max_side, max_side))
        ax.imshow(img)
        if overlay_mask:
            m = Image.open(io.BytesIO(
                read_member(key, member.replace("images/", "masks/", 1)))).convert("L")
            m = m.resize(img.size)
            a = np.array(m)
            ax.imshow(np.ma.masked_where(a == 0, a), cmap="autumn", alpha=0.45)
        ax.set_axis_off()
        ax.set_title(caption(r), fontsize=7.5, color=MUTED, loc="left")
    fig.suptitle(title, fontsize=12, fontweight="bold", color=INK, y=1.04)
    fig.tight_layout()
    plt.show()
    return fig

## 5. Evocargo: авторская разметка

Авторы **не давали классов вообще**. Их аннотация — это сегментация: полигоны
в формате VIA (каталог `json/`, только для пяти съёмок из восьми) и бинарная
маска на каждый из 8190 кадров (каталог `masks/`).

То есть единственный авторский факт про кадр: какая часть его площади занята
каплями. Наша метка `raindrops` выведена из непустоты этой маски — и это
единственная интерпретация, которую мы себе позволили.

In [ ]:
evo = df[df.source == "evocargo_raindrops"].copy()

# mask_nonempty посчитан при заезде: есть ли в авторской маске хоть один пиксель
per_seq = (evo.groupby("sequence_id")
              .agg(кадров=("frame_uid", "size"),
                   с_маской=("mask_nonempty", lambda x: int(x.fillna(False).sum())),
                   разрешение=("width", lambda s: f"{s.iloc[0]}x{evo.loc[s.index[0], 'height']}"),
                   резкость_медиана=("sharpness", lambda s: round(s.median())),
                   яркость_медиана=("luma_mean", lambda s: round(s.mean())))
              .assign(доля_с_маской=lambda t: (100 * t.с_маской / t.кадров).round(1)))
display(per_seq)
print("авторская разметка есть у всех кадров: масок", int(evo.mask_nonempty.notna().sum()),
      "из", len(evo))

### Кадры с непустой авторской маской

In [ ]:
SEED = 3
with_mask = (evo[evo.mask_nonempty == True]
             .groupby("sequence_id", group_keys=False)
             .apply(lambda g: g.sample(1, random_state=SEED))
             .sample(3, random_state=SEED))
fig_evo_pos = grid(with_mask, "Evocargo: авторская маска показана поверх кадра",
                   overlay_mask=True)

### Кадры, у которых авторская маска пуста

In [ ]:
no_mask = (evo[evo.mask_nonempty == False]
           .groupby("sequence_id", group_keys=False)
           .apply(lambda g: g.sample(1, random_state=SEED))
           .sample(3, random_state=SEED))
fig_evo_neg = grid(no_mask, "Evocargo: маска пуста — авторы капель не разметили")

## 6. CADC: авторская разметка

Здесь наоборот: пиксельной разметки помех нет вовсе. Авторы дают **атрибуты на
заезд целиком** в файле `cadc_dataset_route_stats.csv` из девкита, плюс трёхмерные
кубоиды объектов, которые мы не качали.

Ниже их таблица как есть, по тем двадцати заездам, что мы взяли. Колонки
авторские, ничего не переименовано:

* `Snow points removed` — сколько точек лидара отфильтровано как снег; косвенная
  мера интенсивности снегопада;
* `Road snow cover` — покрыта ли дорога снегом;
* `Cam 00 lens snow cover` — налип ли снег на переднюю камеру;
* `Dataset Type` — авторское деление train/test.

**Это аннотация на съёмку, а не на кадр.** Внутри одного заезда кадры разные.

In [ ]:
CADC_AUTHOR = pd.DataFrame([{"sequence_id": "2018_03_06_0005", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 396, "Road snow cover": "None", "Cam 00 lens snow cover": "Partial"}, {"sequence_id": "2018_03_06_0006", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 500, "Road snow cover": "None", "Cam 00 lens snow cover": "Partial"}, {"sequence_id": "2018_03_06_0008", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 265, "Road snow cover": "None", "Cam 00 lens snow cover": "Partial"}, {"sequence_id": "2018_03_06_0009", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 235, "Road snow cover": "None", "Cam 00 lens snow cover": "Partial"}, {"sequence_id": "2018_03_06_0010", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 75, "Road snow cover": "None", "Cam 00 lens snow cover": "Partial"}, {"sequence_id": "2018_03_07_0001", "Dataset Type": "TRAIN", "Frame Count": 50, "Snow points removed": 557, "Road snow cover": "None", "Cam 00 lens snow cover": "Partial"}, {"sequence_id": "2018_03_07_0002", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 472, "Road snow cover": "None", "Cam 00 lens snow cover": "Partial"}, {"sequence_id": "2018_03_07_0005", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 100, "Road snow cover": "None", "Cam 00 lens snow cover": "Partial"}, {"sequence_id": "2018_03_07_0006", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 42, "Road snow cover": "None", "Cam 00 lens snow cover": "Partial"}, {"sequence_id": "2018_03_07_0007", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 182, "Road snow cover": "None", "Cam 00 lens snow cover": "Partial"}, {"sequence_id": "2019_02_27_0011", "Dataset Type": "TRAIN", "Frame Count": 67, "Snow points removed": 971, "Road snow cover": "Covered", "Cam 00 lens snow cover": "None"}, {"sequence_id": "2019_02_27_0037", "Dataset Type": "TRAIN", "Frame Count": 58, "Snow points removed": 1176, "Road snow cover": "Covered", "Cam 00 lens snow cover": "None"}, {"sequence_id": "2019_02_27_0041", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 1066, "Road snow cover": "Covered", "Cam 00 lens snow cover": "None"}, {"sequence_id": "2019_02_27_0043", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 1375, "Road snow cover": "Covered", "Cam 00 lens snow cover": "None"}, {"sequence_id": "2019_02_27_0046", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 884, "Road snow cover": "Covered", "Cam 00 lens snow cover": "None"}, {"sequence_id": "2019_02_27_0049", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 937, "Road snow cover": "Covered", "Cam 00 lens snow cover": "None"}, {"sequence_id": "2019_02_27_0051", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 1105, "Road snow cover": "Covered", "Cam 00 lens snow cover": "None"}, {"sequence_id": "2019_02_27_0068", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 1327, "Road snow cover": "Covered", "Cam 00 lens snow cover": "None"}, {"sequence_id": "2019_02_27_0070", "Dataset Type": "TRAIN", "Frame Count": 50, "Snow points removed": 1443, "Road snow cover": "Covered", "Cam 00 lens snow cover": "None"}, {"sequence_id": "2019_02_27_0078", "Dataset Type": "TRAIN", "Frame Count": 100, "Snow points removed": 932, "Road snow cover": "Covered", "Cam 00 lens snow cover": "None"}])
display(CADC_AUTHOR)
print("покрытие:", CADC_AUTHOR["Frame Count"].sum(), "кадров по данным авторов")

### Заезды, где авторы отметили снег на объективе

In [ ]:
cadc = df[df.source == "cadc"].merge(CADC_AUTHOR, on="sequence_id", how="left")

partial = (cadc[cadc["Cam 00 lens snow cover"] == "Partial"]
           .groupby("sequence_id", group_keys=False)
           .apply(lambda g: g.sample(1, random_state=SEED))
           .sample(3, random_state=SEED))
fig_cadc_partial = grid(partial, "CADC: Cam 00 lens snow cover = Partial")

### Заезды, где авторы отметили чистый объектив

In [ ]:
clean_lens = (cadc[cadc["Cam 00 lens snow cover"] == "None"]
              .groupby("sequence_id", group_keys=False)
              .apply(lambda g: g.sample(1, random_state=SEED))
              .sample(3, random_state=SEED))
fig_cadc_none = grid(clean_lens, "CADC: Cam 00 lens snow cover = None")

## 7. Размер и качество

Разрешение здесь — свойство источника, а не кадра: внутри одной съёмки оно не
меняется. Резкость же меняется сильно, и по её распределению выбирается порог
для метки «смаз».

Обратите внимание на средний вес файла. У Evocargo кадры 1920x1080 весят около
0.27 МБ, а 1280x1024 — примерно 1.5 МБ, хотя пикселей в первых **больше**. PNG
сжимает без потерь, значит в кадрах 1920x1080 просто меньше мелких деталей и
шума: другая камера или другая обработка. Это ещё один признак, по которому
съёмки различимы без всякого взгляда на капли — то же семейство утечки, что и
разрешение.

In [ ]:
size_tbl = (df.assign(разрешение=df.width.astype(str) + "x" + df.height.astype(str))
              .groupby(["source", "разрешение"])
              .agg(кадров=("frame_uid", "size"),
                   средний_вес_МБ=("bytes", lambda s: round(s.mean() / 1024**2, 2)),
                   резкость_p05=("sharpness", lambda s: round(np.percentile(s, 5))),
                   резкость_медиана=("sharpness", lambda s: round(s.median())),
                   резкость_p95=("sharpness", lambda s: round(np.percentile(s, 95))),
                   яркость=("luma_mean", lambda s: round(s.mean()))))
display(size_tbl)

fig_q, ax = plt.subplots(figsize=(9, 3.4))
for name, color in (("evocargo_raindrops", BLUE), ("cadc", ORANGE)):
    v = df[df.source == name].sharpness
    if len(v):
        ax.hist(v, bins=60, range=(0, 1500), alpha=0.65, color=color, label=name)
ax.set_xlabel("резкость (дисперсия лапласиана)")
ax.set_ylabel("кадров")
ax.legend(frameon=False)
ax.set_title("Распределение резкости по источникам", loc="left",
             fontweight="bold", color=INK)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
fig_q.tight_layout()
plt.show()

## 8. Авторское против нашего

Только теперь смотрим на свои метки — и рядом на то, из чего они выведены.

In [ ]:
mapping = pd.DataFrame([
    {"источник": "Evocargo", "авторская аннотация": "маска капель непуста",
     "наша метка": "raindrops", "статус": "прямое следствие"},
    {"источник": "Evocargo", "авторская аннотация": "маска пуста",
     "наша метка": "clean", "статус": "прямое следствие"},
    {"источник": "CADC", "авторская аннотация": "Cam 00 lens snow cover = Partial (2018)",
     "наша метка": "raindrops", "статус": "исправлено: было snowfall+soiling"},
    {"источник": "CADC", "авторская аннотация": "Cam 00 lens snow cover = None (2019)",
     "наша метка": "snowfall", "статус": "по Road snow cover = Covered"},
])
display(mapping)

ours = df.assign(наша=df.labels.apply(lambda l: "+".join(sorted(l)) or "clean"))
display(ours.groupby(["source", "наша"]).size().rename("кадров").to_frame())

print("""
Где мы ошиблись и почему это видно на картинках выше:

  Заезды 2018 года авторы пометили Road snow cover = None — дорога голая. Мы
  всё равно поставили им snowfall, рассуждая «зимний датасет». На кадрах в
  разделе 6 видно мокрый асфальт и воду на стекле, снега нет.

  Snow points removed у них 42..557 против 884..1443 у 2019 года. Авторская
  таблица предупреждала, мы не посмотрели.
""")

## 9. В ClearML

In [ ]:
import subprocess, sys

LIBS = os.path.expanduser("~/dslibs")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "--target", LIBS, "clearml"])
if LIBS not in sys.path:
    sys.path.insert(0, LIBS)

# ключи хранилища ClearML берёт из своего конфига, а не из наших переменных
tpl = """sdk {
    aws {
        s3 {
            region: "ru-central1"
            credentials: [
                { host: "storage.yandexcloud.net:443"
                  key: "@KEY@"
                  secret: "@SECRET@"
                  secure: true }
            ]
        }
    }
}
"""
conf = os.path.expanduser("~/clearml.conf")
with open(conf, "w") as fh:
    fh.write(tpl.replace("@KEY@", os.environ["S3_KEY"])
                .replace("@SECRET@", os.environ["S3_SECRET"]))
os.chmod(conf, 0o600)

from clearml import Task

os.environ.setdefault("CLEARML_WEB_HOST",   "https://app.occlusionnet.duckdns.org")
os.environ.setdefault("CLEARML_API_HOST",   "https://api.occlusionnet.duckdns.org")
os.environ.setdefault("CLEARML_FILES_HOST", "https://files.occlusionnet.duckdns.org")

task = Task.init(project_name="OcclusionNet", task_name="source-labels",
                 task_type=Task.TaskTypes.data_processing,
                 output_uri=f"s3://storage.yandexcloud.net:443/{BUCKET}")
log = task.get_logger()

log.report_table("Авторская разметка", "Evocargo по съёмкам", table_plot=per_seq.reset_index())
log.report_table("Авторская разметка", "CADC по заездам",     table_plot=CADC_AUTHOR)
log.report_table("Размер и качество",   "По источникам",       table_plot=size_tbl.reset_index())
log.report_table("Соответствие",        "Авторское -> наше",   table_plot=mapping)

for name, n in ours.groupby("наша").size().items():
    log.report_single_value(f"кадров_{name}", int(n))
log.report_single_value("кадров_всего", len(df))

for title, fig in (("Evocargo с маской", fig_evo_pos),
                   ("Evocargo без маски", fig_evo_neg),
                   ("CADC lens Partial", fig_cadc_partial),
                   ("CADC lens None", fig_cadc_none),
                   ("Резкость", fig_q)):
    if fig is not None:
        log.report_matplotlib_figure("Примеры", title, figure=fig, report_image=True)

task.close()
print("готово:", task.get_output_log_web_page())